# FoodScholar — Building the Graph

Two ways to read this notebook:

- **§1 Quickstart** — the 5-line happy path with `FoodScholar.in_memory()`. Run this if you just want to see the library work.
- **§2–§12 Walk-through** — a phase-by-phase build of the three-layer graph from raw chunks. Sections for phases that are not yet implemented are marked `[STUB]` and live next to the contract they will eventually call. As real implementations land, replace the stub cell in place — the surface around it stays stable.

**Current status (v0.1.0):** Foundation only. The in-memory stores work; annotate / layer_a / layer_b / layer_c / retrieval are stubs.

Run from the repo root (with `conda activate foodscholar`):
```bash
jupyter notebook notebooks/build_graph.ipynb
```

## Setup

Make the package importable when running the notebook from `notebooks/`. If you `pip install -e .` into a conda env you can skip this cell.

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import foodscholar
print("foodscholar", foodscholar.__version__)
print("repo root  ", REPO_ROOT)

: 

In [ ]:
from foodscholar.logging import configure_logging, get_logger

configure_logging(level="INFO")
log = get_logger("notebook")

## 1. Quickstart — the 5-line happy path

If all you want is to see the library work, this is the public API. Everything below it is the longer phase-by-phase walk-through.

Phase methods that haven't been implemented yet (`annotate`, `build_layer_a`, ...) raise `NotImplementedError` with a clear message — the surface stays correct as the milestones land.

In [ ]:
from foodscholar import FoodScholar
from foodscholar.io.chunk import Chunk

fs = FoodScholar.in_memory()

fs.upsert_chunks([
    Chunk(chunk_id="c1", text="Mediterranean diet reduces cardiovascular risk.",
          source_doc_id="d1", source_type="abstract", section_type="abstract"),
])

fs.info()

In [ ]:
# Annotate now ships, so a real fs.build() needs an ontology attached.
# Without one, fs.build() raises a clear error pointing you at the next step.
try:
    fs.build()
except (RuntimeError, NotImplementedError) as e:
    print("deferred:", e)

## 2. Walk-through — phase by phase

The Quickstart used the in-memory factory. This walk-through covers the same path but stops at every phase so you can see what each step contributes. Sections labeled `[STUB]` correspond to phases not yet implemented in v0.1.0; they perform the equivalent operation manually via `fs.graph.*` so the surface stays stable as real implementations land.

## 3. Load configuration

We use the example config but override both backends to `memory` so the notebook runs without Elasticsearch or Neo4j. Swap to `elastic` / `neo4j` once those adapters land.

In [ ]:
import yaml
from foodscholar import FoodScholar, FoodScholarConfig

raw = yaml.safe_load((REPO_ROOT / "config.example.yaml").read_text())
raw["storage"]["chunk_store"] = {"backend": "memory"}
raw["storage"]["graph_store"] = {"backend": "memory"}

cfg = FoodScholarConfig.model_validate(raw)
fs = FoodScholar.from_config(cfg)
fs.info()

## 4. Load chunks

Pass a `chunks.parquet` (or `.jsonl`) path to `fs.load_chunks()`. Here we build a tiny synthetic corpus so the notebook is self-contained.

In [ ]:
from foodscholar.io.chunk import Chunk

mini = [
    Chunk(chunk_id="c1", text="Mediterranean diet rich in olive oil reduces cardiovascular risk.",
          source_doc_id="doc-1", source_type="abstract", section_type="abstract", year=2024),
    Chunk(chunk_id="c2", text="Whole grain consumption is associated with lower mortality.",
          source_doc_id="doc-2", source_type="abstract", section_type="results", year=2023),
    Chunk(chunk_id="c3", text="Peanut allergy management guidelines for paediatric populations.",
          source_doc_id="doc-3", source_type="guide", section_type="guideline", year=2022),
    Chunk(chunk_id="c4", text="Iron-rich foods include legumes, red meat, and fortified cereals.",
          source_doc_id="doc-4", source_type="textbook", section_type="textbook", year=2021),
    Chunk(chunk_id="c5", text="Plant-based dietary patterns are linked to improved metabolic markers.",
          source_doc_id="doc-5", source_type="abstract", section_type="discussion", year=2024),
]
fs.upsert_chunks(mini)
len(fs.chunk_store.scan())

## 5. Load the FoodOn ontology

**Brief §2:** v1 uses **FoodOn only**, loaded with pronto and `import_depth=0`. The ontology drives the linker (mention → `foodon_id`), the Layer A backbone (frequency-weighted ancestor propagation), and Layer C card prompts.

Two ways to wire one in:
1. **Production:** point `cfg.ontology.foodon_path` at a real FoodOn `.owl` file. `fs.ontology` lazily loads it and caches to Parquet.
2. **Notebook / test:** build an in-memory `FoodOnAPI` from a small `OntologyTerm` list and call `fs.attach_ontology(api)`. We use a tiny synthetic ontology here so the notebook stays self-contained.

In [ ]:
from foodscholar import FoodOnAPI
from foodscholar.ontology import load_ontology

# Use the test fixture so the notebook never needs the ~100MB FoodOn release.
MINI_FOODON = REPO_ROOT / "tests" / "fixtures" / "mini_foodon.obo"
fs.attach_ontology(FoodOnAPI(load_ontology(MINI_FOODON), prefix_filter=None))

ont = fs.ontology
print(f"loaded {len(ont)} terms")
print("olive oil ->", ont.name_to_id("olive oil"))
print("EVOO ->", ont.name_to_id("EVOO"))
print("olive oil ancestors:", [ont.id_to_label(a) for a in ont.id_to_ancestors("TEST:0000008")])
print("plant food descendants:", [ont.id_to_label(d) for d in ont.id_to_descendants("TEST:0000002")])

## 6. Annotate

**Brief §2 / §12 step 9 — shipped.** SciFoodNER (or `KeywordNER` over the ontology) extracts food mentions, the three-tier linker (`lexical_exact` → `lexical_fuzzy` → `dense`) maps each mention to a FoodOn id, and the configured embedder produces a vector per chunk (SPECTER2 for abstracts, BGE-large for textbook/guide via `SourceTypeRouter`).

The full phase is one line:

In [ ]:
meta = fs.annotate()
print("artifact:", meta.artifact_id, "records:", meta.record_count)

for cid in ["c1", "c2", "c3", "c4", "c5"]:
    c = fs.graph.chunk(cid)
    print(f"  {cid}: mentions={[m.text for m in c.mentions]}, foodon_ids={c.foodon_ids}")

### Probe the linker

`fs.linker.dry_run(text)` is the single-string variant of the link step — useful for sanity-checking how surface forms map to FoodOn ids. The three tiers (`lexical_exact`, `lexical_fuzzy`, `dense`) are visible in `link.method`.

In [ ]:
for q in ["olive oil", "EVOO", "olives", "oliv oil", "evo", "apples", "Arachis hypogaea", "quinoa"]:
    link = fs.linker.dry_run(q)
    if link:
        label = fs.ontology.id_to_label(link.ontology_id)
        print(f"  {q:20s} -> {link.ontology_id} ({label}) [{link.method}, {link.confidence:.2f}]")
    else:
        print(f"  {q:20s} -> None")

## 7. `[STUB]` Layer A — Backbone

**Brief:** project FoodOn down to a corpus-adaptive subset via frequency-weighted ancestor propagation, prune (min-support, depth cap, single-child collapse, blacklist), then merge with curated top-level facets.

**Today:** hand-curated three-shelf backbone using `fs.graph.add_shelf(...)`. Will be replaced by `fs.build_layer_a()`.

In [ ]:
# Layer A will eventually project FoodOn into the backbone automatically.
# For now we hand-pick three shelves from the ontology so the surface lines up.

# olive oil's plant-food ancestor becomes the foods root
foods_root_id = next(
    a for a in fs.ontology.id_to_ancestors("TEST:0000008")
    if fs.ontology.id_to_label(a) == "plant food"
)
fs.graph.add_shelf(shelf_id="s-foods", label=fs.ontology.id_to_label(foods_root_id),
                   facet="foods", depth=0, foodon_id=foods_root_id)
fs.graph.add_shelf(shelf_id="s-med", label="Mediterranean diet",
                   facet="dietary_patterns", depth=1, parent_shelf_id="s-foods")
fs.graph.add_shelf(shelf_id="s-allergy", label="Food allergies",
                   facet="allergies", depth=1)

fs.graph.summary()

## 8. Attach chunks to shelves

Multi-label: a chunk attaches to every shelf whose FoodOn ancestor it touches. `fs.graph.attach_chunks(...)` updates both the graph edges **and** the denormalized `shelf_ids` on each chunk — no need to mirror state by hand. Idempotent, so re-running is safe.

In [ ]:
fs.graph.attach_chunks(["c1", "c2", "c5"], shelf="s-med")
fs.graph.attach_chunks(["c3"],             shelf="s-allergy")
fs.graph.attach_chunks(["c4"],             shelf="s-foods")

for h in fs.graph.shelves():
    print(f"  {h.shelf_id:12s} -> chunks={[c.chunk_id for c in h.chunks()]}")

## 9. `[STUB]` Layer B — Themes per shelf

**Brief:** Leiden community detection on the per-shelf embedding graph (HDBSCAN fallback, BERTopic for prototyping). Themes can attach to multiple shelves.

**Today:** two hand-defined themes on `s-med`. Will be replaced by `fs.build_layer_b()`.

In [ ]:
fs.graph.add_theme(theme_id="t-olive", label="Olive oil cardiovascular benefits",
                   shelf_ids=["s-med"], discovered_by="leiden", discovery_version="notebook-v1")
fs.graph.add_theme(theme_id="t-plant", label="Plant-based metabolic markers",
                   shelf_ids=["s-med"], discovered_by="leiden", discovery_version="notebook-v1")

fs.graph.attach_chunks(["c1"], theme="t-olive")
fs.graph.attach_chunks(["c5"], theme="t-plant")

for t in fs.graph.shelf("s-med").themes():
    print(f"  {t.theme_id:10s} {t.label}  cites={[c.chunk_id for c in t.chunks()]}")

## 10. `[STUB]` Layer C — Write-up cards

**Brief:** LLM produces a card per shelf and per theme — summary, tip, evidence-quality note, optional controversy/confidence, and **citations to every chunk a claim draws from**. Grounding check enforces traceability.

**Today:** two mock cards added via `fs.graph.add_card(...)`. Will be replaced by `fs.build_layer_c()`.

In [ ]:
from foodscholar.io.graph import Card

fs.graph.add_card(Card(
    card_id="card-s-med", target_id="s-med", target_type="shelf",
    title="Mediterranean diet",
    summary="A dietary pattern with strong cardiovascular evidence; emphasizes olive oil, whole grains, and plant foods.",
    tip="Replace butter with extra-virgin olive oil as the primary cooking fat.",
    evidence_quality="high",
    cited_chunk_ids=["c1", "c2", "c5"],
    llm_model=fs.llm.model_id, prompt_version="v1",
))
fs.graph.add_card(Card(
    card_id="card-t-olive", target_id="t-olive", target_type="theme",
    title="Olive oil and cardiovascular health",
    summary="Regular consumption of extra-virgin olive oil is associated with reduced cardiovascular risk.",
    evidence_quality="medium",
    cited_chunk_ids=["c1"],
    llm_model=fs.llm.model_id, prompt_version="v1",
))

# Grounding check — every cited chunk must resolve.
for card_handle in [fs.graph.shelf("s-med").card(), fs.graph.theme("t-olive").card()]:
    missing = [cid for cid in card_handle.cited_chunk_ids if fs.graph.chunk(cid) is None]
    assert not missing, f"card {card_handle.card_id} cites unknown chunks: {missing}"
    print(card_handle, "-> ok")

## 11. Inspect the graph

Every read is a method on `fs.graph` or one of its handles. Pydantic models stay accessible via `handle.model` if you need to serialize.

In [ ]:
print("Summary:", fs.graph.summary())

print("\nShelves:")
for h in fs.graph.shelves():
    parent = f" (parent={h.parent_shelf_id})" if h.parent_shelf_id else ""
    print(f"  [{h.facet:18s} d={h.depth}] {h.shelf_id:12s} {h.label}{parent}")

print("\nThemes on Mediterranean shelf:")
for t in fs.graph.shelf("s-med").themes():
    print(f"  {t.theme_id:10s} {t.label}")

print("\nWalk a card -> its cited chunks:")
card = fs.graph.theme("t-olive").card()
for c in card.cited_chunks():
    print(f"  {c.chunk_id}: {c.text[:60]}...")

## 12. `[STUB]` Query — retrieval pipeline

**Brief (Section 14):** parse → resolve to shelves → resolve to themes → fetch pre-generated cards → hybrid chunk retrieval → LLM synthesize → grounding check.

**Today:** `fs.query(...)` raises NotImplementedError. We exercise the retrieval *shape* by composing the steps manually with `fs.graph.search(...)` and `fs.graph.theme(...).card()`.

In [ ]:
from foodscholar.retrieval import Answer

query = "olive oil cardiovascular health"
activated_shelves = ["s-med"]
activated_themes  = ["t-olive"]

hits = fs.graph.search(query, theme="t-olive", k=5)
card_handles = [fs.graph.shelf(s).card() for s in activated_shelves]
card_handles += [fs.graph.theme(t).card() for t in activated_themes]
matched_cards = [ch for ch in card_handles if ch is not None]

answer = Answer(
    text="; ".join(ch.summary for ch in matched_cards),
    tips=[ch.tip for ch in matched_cards if ch.tip],
    cited_chunks=[h.chunk_id for h in hits],
    cited_cards=[ch.card_id for ch in matched_cards],
    activated_shelves=activated_shelves,
    activated_themes=activated_themes,
    grounding_passed=True,
    llm_model="none (stub)",
    prompt_version="v0",
)
print(answer.model_dump_json(indent=2))

## 13. Stamp an artifact

Every build should produce an `ArtifactMeta` so downstream phases can verify provenance and re-runs are reproducible.

In [ ]:
from foodscholar import make_artifact_meta

meta = make_artifact_meta(phase="notebook-build", config=fs.config, record_count=len(mini))
print(meta.model_dump_json(indent=2))

## Next milestones

Each milestone replaces a `[STUB]` cell above with a single facade call:

| Section | Phase            | Replaces cell                              |
|---------|------------------|---------------------------------------------|
| §7      | layer_a          | `fs.build_layer_a()`                        |
| §9      | layer_b          | `fs.build_layer_b()`                        |
| §10      | layer_c          | `fs.build_layer_c()`                        |
| §12     | retrieval        | `fs.query("...")`                          |

Implementation order is fixed in BRIEF §12. **Maintain this notebook in lockstep** — each milestone replaces its stub cell with a one-line phase call. The narrative around it stays the same.